In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlscontenedor")

In [0]:

container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

In [0]:
ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/"
items = ruta + "items.csv"


In [0]:
producto_schema = StructType(fields=[StructField("ID_ITEM", StringType(), False),
                                  StructField("PRODUCT", StringType(), False),
                                  StructField("CATEGORY", StringType(), False)
                                  ])


In [0]:
%python
df_productos = (spark.read.format("csv")
    .option("header", "true")    
    .schema(producto_schema)
    .load(items))

In [0]:
df_productos_ingestion_date = df_productos.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
df_productos_final = df_productos_ingestion_date.select(
                                    "ID_ITEM",
                                    "PRODUCT",
                                    "CATEGORY",
                                    "INGESTION_DATE"
                                )


In [0]:
df_productos_final.write.mode("overwrite").option("overwriteSchema", "true").insertInto(f"{catalogo}.{esquema}.productos")

/*spark.conf.set(f"fs.azure.account.key.{storageName}.dfs.core.windows.net", connectionString.strip())

df_ventas = spark.read.option('header', True)\
                           .option('inferSchema', True)\
                           .csv("abfss://raw-retail@adlsproject.dfs.core.windows.net/orders.csv")

display(df_ventas)

select * from catalog_dev.bronze.productos;